# 4주차 Gemma RAG — GPU 통합본

**이 파일 하나만 사용하세요. 이전 노트북·수정 파일·CPU ZIP을 섞지 마세요.**

질문 5개 × 기준선/재순위화/질문 분해 = 15개 결과를 같은 기준으로 비교합니다. 자료 3개·17쪽·44청크가 포함됩니다. 모델명 태그 확인과 저장 경로 문제를 반영했습니다.

Colab에서 **런타임 → 런타임 유형 변경 → GPU**로 연결한 뒤 1~9번을 순서대로 실행하세요. GPU가 배정되지 않으면 코드로 배정을 강제할 수 없습니다. 2번 자료 복원 셀은 길지만 접혀 있으며 수정 없이 실행하면 됩니다. 첫 셀은 저장 위치와 GPU만 확인합니다.

작성 단계의 검증은 오프라인 검사입니다. 실제 Colab GPU 모델 실행은 5번에서 확인하며, 성능 결과는 6~8번 실행 후 판단합니다.


## 1. GPU 확인과 저장 위치 선택

기본은 Google Drive 보관입니다. 연결을 허용하세요. 기존 과제 폴더를 지우지 않고 새 폴더에 저장합니다.


In [ ]:
USE_DRIVE=True
EXPERIMENT_FOLDER='gemma_rag_gpu_clean_v1'
import subprocess,shutil
from pathlib import Path
if not Path('/content').is_dir(): raise RuntimeError('Google Colab에서 실행하세요.')
gpu=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True) if shutil.which('nvidia-smi') else None
if gpu is None or gpu.returncode!=0:
    raise RuntimeError('실제 GPU가 없습니다. GPU 런타임이 배정된 뒤 실행하세요. 이 단계에서는 파일/모델을 설치하지 않았습니다.')
print('할당된 GPU:',gpu.stdout.strip())
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT=(Path('/content/drive/MyDrive/AI_AGENT/week04') if USE_DRIVE else Path('/content/gemma_rag_week04'))/EXPERIMENT_FOLDER
PROJECT.mkdir(parents=True,exist_ok=True)
Path('/content/week04_gpu_project_path.txt').write_text(str(PROJECT))
print('이 노트북의 저장 위치:',PROJECT)


## 2. 내장 자료·코드 복원

추가 파일을 업로드하지 않습니다. 이 셀은 접힌 상태 그대로 실행하세요. 다른 셀에 내용을 복사하지 마세요.


In [ ]:
#@title 2. 내장 자료·코드 복원 — 수정 없이 실행
import json,subprocess,sys
from pathlib import Path
MARKER=Path('/content/week04_gpu_project_path.txt')
if not MARKER.is_file(): raise RuntimeError('먼저 1번 저장 위치 셀을 실행하세요.')
PROJECT=Path(MARKER.read_text().strip())
if not PROJECT.is_dir(): raise RuntimeError('저장 위치에 접근할 수 없습니다. 1번에서 Drive 연결을 확인하세요.')
PY=Path('/content/week04_env/bin/python')
def run_stage(script,*args,needs_env=True):
    if not (PROJECT/'src'/script).is_file(): raise RuntimeError('2번 자료 복원 셀을 실행하세요.')
    if needs_env and not PY.is_file(): raise RuntimeError('3번 환경 설치 셀을 실행하세요.')
    command=[str(PY) if needs_env else sys.executable,'-u',str(PROJECT/'src'/script),*args]
    p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait(): raise RuntimeError('실행 실패. 위의 실제 오류를 확인하세요. 저장 결과는 보존됩니다.')

BUNDLE_BASE64='...'
BUNDLE_SHA256='8087c9d7b0d6d2ac6a6ec86db3fa89f472332659cb132678eaa91d3a7a11e4dd'
import base64,hashlib,io,zipfile
bundle=base64.b64decode(BUNDLE_BASE64)
if hashlib.sha256(bundle).hexdigest()!=BUNDLE_SHA256: raise RuntimeError('내장 자료 손상')
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for name in archive.namelist():
        p=(PROJECT/name).resolve()
        if not p.is_relative_to(PROJECT.resolve()): raise RuntimeError('잘못된 내장 경로')
        if p.exists() and p.read_bytes()!=archive.read(name) and (PROJECT/'results/latest.json').exists():
            raise RuntimeError('기존 실험과 코드/자료가 다릅니다. 1번 폴더 이름을 바꿔 별도 실험으로 보존하세요.')
    for name in archive.namelist():
        p=PROJECT/name; p.parent.mkdir(parents=True,exist_ok=True)
        if not p.exists() or p.read_bytes()!=archive.read(name): p.write_bytes(archive.read(name))
print('자료와 코드 복원 완료:',PROJECT/'src/experiment.py')


## 3. 패키지·Ollama·모델 설치

첫 설치와 모델 다운로드에는 시간이 걸립니다. 끝에 설치 모델 목록을 표시합니다. `embeddinggemma:latest`도 정상입니다.


In [ ]:
import json,subprocess,sys
from pathlib import Path
MARKER=Path('/content/week04_gpu_project_path.txt')
if not MARKER.is_file(): raise RuntimeError('먼저 1번 저장 위치 셀을 실행하세요.')
PROJECT=Path(MARKER.read_text().strip())
if not PROJECT.is_dir(): raise RuntimeError('저장 위치에 접근할 수 없습니다. 1번에서 Drive 연결을 확인하세요.')
PY=Path('/content/week04_env/bin/python')
def run_stage(script,*args,needs_env=True):
    if not (PROJECT/'src'/script).is_file(): raise RuntimeError('2번 자료 복원 셀을 실행하세요.')
    if needs_env and not PY.is_file(): raise RuntimeError('3번 환경 설치 셀을 실행하세요.')
    command=[str(PY) if needs_env else sys.executable,'-u',str(PROJECT/'src'/script),*args]
    p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait(): raise RuntimeError('실행 실패. 위의 실제 오류를 확인하세요. 저장 결과는 보존됩니다.')

import time,urllib.request
PY=Path('/content/week04_env/bin/python')
def visible(command):
    p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait(): raise RuntimeError('명령 실행 실패. 위 오류를 확인하세요.')
if not PY.exists():
    visible([sys.executable,'-m','venv','--without-pip',str(PY.parent.parent)])
marker=PY.parent.parent/'week04_requirements.json'
requirements=(PROJECT/'requirements.txt').read_text()
if not marker.exists() or marker.read_text()!=requirements:
    visible([sys.executable,'-m','pip','--python',str(PY),'install','--disable-pip-version-check','-q','pip'])
    visible([str(PY),'-m','pip','install','--disable-pip-version-check','-q','torch==2.9.1','--index-url','https://download.pytorch.org/whl/cpu'])
    visible([str(PY),'-m','pip','install','--disable-pip-version-check','-q','-r',str(PROJECT/'requirements.txt')])
    visible([str(PY),'-m','pip','check'])
    visible([str(PY),'-c','from ragas.metrics.collections import Faithfulness; from ragas.metrics import IDBasedContextRecall; import chromadb,torch; from transformers import AutoModelForSequenceClassification; print("패키지 불러오기 성공")'])
    marker.write_text(requirements)

import shutil,os
os.environ['OLLAMA_MAX_LOADED_MODELS']='1'
os.environ['OLLAMA_NUM_PARALLEL']='1'
if not shutil.which('ollama'):
    # Official installer needs zstd on some fresh Colab runtimes.
    if not shutil.which('zstd'):
        visible(['apt-get','update','-qq'])
        visible(['apt-get','install','-y','-qq','zstd'])
    installer=Path('/content/week04_install_ollama.sh')
    installer.write_bytes(urllib.request.urlopen('https://ollama.com/install.sh',timeout=60).read())
    visible(['sh',str(installer)])
def server_ready():
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags',timeout=3)
        return True
    except Exception:
        return False
if not server_ready():
    log=open('/content/week04_ollama.log','a')
    subprocess.Popen(['ollama','serve'],stdout=log,stderr=log,start_new_session=True)
    for _ in range(30):
        if server_ready(): break
        time.sleep(1)
    if not server_ready(): raise RuntimeError('Ollama 시작 실패. /content/week04_ollama.log 확인')
visible(['ollama','pull','gemma3:4b'])
visible(['ollama','pull','embeddinggemma'])
print('저장 위치:',PROJECT)
print('다음은 4번 셀입니다. 이 셀을 재실행해도 완료된 답변을 지우지 않습니다.')

visible(['ollama','list'])


## 4. 질문·기대 답변 확인과 인덱스 준비

평가 기준은 실행 전에 고정합니다. 기대 답변·라벨은 생성 입력에 넣지 않습니다. 모델 확인 시 생략된 태그를 `:latest`와 동일하게 처리합니다.


In [ ]:
import json,subprocess,sys
from pathlib import Path
MARKER=Path('/content/week04_gpu_project_path.txt')
if not MARKER.is_file(): raise RuntimeError('먼저 1번 저장 위치 셀을 실행하세요.')
PROJECT=Path(MARKER.read_text().strip())
if not PROJECT.is_dir(): raise RuntimeError('저장 위치에 접근할 수 없습니다. 1번에서 Drive 연결을 확인하세요.')
PY=Path('/content/week04_env/bin/python')
def run_stage(script,*args,needs_env=True):
    if not (PROJECT/'src'/script).is_file(): raise RuntimeError('2번 자료 복원 셀을 실행하세요.')
    if needs_env and not PY.is_file(): raise RuntimeError('3번 환경 설치 셀을 실행하세요.')
    command=[str(PY) if needs_env else sys.executable,'-u',str(PROJECT/'src'/script),*args]
    p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait(): raise RuntimeError('실행 실패. 위의 실제 오류를 확인하세요. 저장 결과는 보존됩니다.')
from IPython.display import display, Markdown
refs=json.loads((PROJECT/'data/references.json').read_text())
for ref in refs:
    display(Markdown('**'+ref['id']+' — '+ref['answerability']+'**\n\n'+ref['question']+
                     '\n\n기대 답변: '+ref['expected_answer']+'\n\n근거: '+ref['source_notes']))
run_stage('experiment.py','prepare')


## 5. 실제 GPU 소규모 실행 점검

작은 임베딩·Gemma 요청을 실행하고 `ollama ps`에서 GPU 사용을 확인합니다. preflight 기록으로 분리하며 질문 5개의 평가에는 포함하지 않습니다. 통과 후 전체 비교를 실행하세요.


In [ ]:
import json,subprocess,sys
from pathlib import Path
MARKER=Path('/content/week04_gpu_project_path.txt')
if not MARKER.is_file(): raise RuntimeError('먼저 1번 저장 위치 셀을 실행하세요.')
PROJECT=Path(MARKER.read_text().strip())
if not PROJECT.is_dir(): raise RuntimeError('저장 위치에 접근할 수 없습니다. 1번에서 Drive 연결을 확인하세요.')
PY=Path('/content/week04_env/bin/python')
def run_stage(script,*args,needs_env=True):
    if not (PROJECT/'src'/script).is_file(): raise RuntimeError('2번 자료 복원 셀을 실행하세요.')
    if needs_env and not PY.is_file(): raise RuntimeError('3번 환경 설치 셀을 실행하세요.')
    command=[str(PY) if needs_env else sys.executable,'-u',str(PROJECT/'src'/script),*args]
    p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait(): raise RuntimeError('실행 실패. 위의 실제 오류를 확인하세요. 저장 결과는 보존됩니다.')
run_stage('preflight.py')


## 6. 세 구성 비교 실행

5문항×3구성입니다. 완료 결과는 재사용하고 매 시도 직후 저장합니다. 재순위화 모델은 CPU에서 실행하고 답변·질문 분해는 GPU Gemma를 사용합니다. B에 A를 함께 적용하지 않습니다.


In [ ]:
import json,subprocess,sys
from pathlib import Path
MARKER=Path('/content/week04_gpu_project_path.txt')
if not MARKER.is_file(): raise RuntimeError('먼저 1번 저장 위치 셀을 실행하세요.')
PROJECT=Path(MARKER.read_text().strip())
if not PROJECT.is_dir(): raise RuntimeError('저장 위치에 접근할 수 없습니다. 1번에서 Drive 연결을 확인하세요.')
PY=Path('/content/week04_env/bin/python')
def run_stage(script,*args,needs_env=True):
    if not (PROJECT/'src'/script).is_file(): raise RuntimeError('2번 자료 복원 셀을 실행하세요.')
    if needs_env and not PY.is_file(): raise RuntimeError('3번 환경 설치 셀을 실행하세요.')
    command=[str(PY) if needs_env else sys.executable,'-u',str(PROJECT/'src'/script),*args]
    p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait(): raise RuntimeError('실행 실패. 위의 실제 오류를 확인하세요. 저장 결과는 보존됩니다.')
RETRY_ERRORS = False
run_stage('experiment.py','run',*(['--retry-errors'] if RETRY_ERRORS else []))


## 7. 공통 기준 자동 평가

근거 Recall·필수요소·연도 표기·Faithfulness를 같은 기준으로 검사합니다. 전체 정답률은 아닙니다. 적용 불가·평가 오류는 사유와 함께 보존합니다.


In [ ]:
import json,subprocess,sys
from pathlib import Path
MARKER=Path('/content/week04_gpu_project_path.txt')
if not MARKER.is_file(): raise RuntimeError('먼저 1번 저장 위치 셀을 실행하세요.')
PROJECT=Path(MARKER.read_text().strip())
if not PROJECT.is_dir(): raise RuntimeError('저장 위치에 접근할 수 없습니다. 1번에서 Drive 연결을 확인하세요.')
PY=Path('/content/week04_env/bin/python')
def run_stage(script,*args,needs_env=True):
    if not (PROJECT/'src'/script).is_file(): raise RuntimeError('2번 자료 복원 셀을 실행하세요.')
    if needs_env and not PY.is_file(): raise RuntimeError('3번 환경 설치 셀을 실행하세요.')
    command=[str(PY) if needs_env else sys.executable,'-u',str(PROJECT/'src'/script),*args]
    p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait(): raise RuntimeError('실행 실패. 위의 실제 오류를 확인하세요. 저장 결과는 보존됩니다.')
RUN_FAITHFULNESS = True
RETRY_EVALUATION_ERRORS = False
args=[]
if not RUN_FAITHFULNESS: args.append('--skip-faithfulness')
if RETRY_EVALUATION_ERRORS: args.append('--retry-errors')
run_stage('evaluation.py',*args)
latest=json.loads((PROJECT/'results/latest.json').read_text())
from IPython.display import display, Markdown
display(Markdown((PROJECT/latest['relative_path']/'report.md').read_text()))


## 8. 사람 판정과 대표 사례 분석

원문·실제 컨텍스트·답변을 비교하고 15개 판정과 이유를 저장하세요. 대표 실패와 성공 항목의 회귀, 두 기법의 유지/수정/제거 판단을 작성합니다.


In [ ]:
import json,subprocess,sys
from pathlib import Path
MARKER=Path('/content/week04_gpu_project_path.txt')
if not MARKER.is_file(): raise RuntimeError('먼저 1번 저장 위치 셀을 실행하세요.')
PROJECT=Path(MARKER.read_text().strip())
if not PROJECT.is_dir(): raise RuntimeError('저장 위치에 접근할 수 없습니다. 1번에서 Drive 연결을 확인하세요.')
PY=Path('/content/week04_env/bin/python')
def run_stage(script,*args,needs_env=True):
    if not (PROJECT/'src'/script).is_file(): raise RuntimeError('2번 자료 복원 셀을 실행하세요.')
    if needs_env and not PY.is_file(): raise RuntimeError('3번 환경 설치 셀을 실행하세요.')
    command=[str(PY) if needs_env else sys.executable,'-u',str(PROJECT/'src'/script),*args]
    p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait(): raise RuntimeError('실행 실패. 위의 실제 오류를 확인하세요. 저장 결과는 보존됩니다.')
import html
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
RUN=PROJECT/json.loads((PROJECT/'results/latest.json').read_text())['relative_path']
def read_json(path,default):
    return json.loads(path.read_text()) if path.exists() else default
records=read_json(RUN/'records.json',[])
refs={r['id']:r for r in read_json(PROJECT/'data/references.json',[])}
evaluations={e['key']:e for e in read_json(RUN/'evaluations.json',[])}
previous=read_json(RUN/'human_review.json',{'reviews':[],'decisions':{},'analysis':''})
old={r['key']:r for r in previous['reviews']}
forms=[]
pages=read_json(PROJECT/'data/pages.json',[])
page_view=widgets.Accordion(children=[widgets.HTML('<pre style="white-space:pre-wrap">'+html.escape(p['raw_text'])+'</pre>') for p in pages])
for i,p in enumerate(pages): page_view.set_title(i,p['doc_id']+' PDF '+str(p['page'])+'쪽')
display(Markdown('**원문 추출문: 문서 ID와 페이지를 확인하세요. 필요하면 Drive의 data/pdf 원본도 확인하세요.**'))
display(page_view)
for qid,ref in refs.items():
    display(Markdown('**'+qid+'** '+ref['question']+'\n\n기대 답변: '+ref['expected_answer']+'\n\n근거: '+ref['source_notes']))
    panels=[]; titles=[]
    for rec in [r for r in records if r['question_id']==qid]:
        output=widgets.Output()
        with output:
            display(Markdown('실행: **'+rec['status']+'** / '+str(round(rec['wall_seconds'],2))+'초\n\n```text\n'+(rec['answer'] or str(rec['error']))+'\n```'))
            for name,m in evaluations.get(rec['key'],{}).get('metrics',{}).items():
                print(name,':',m['value'] if m['value'] is not None else m['status'],'—',m['reason'])
        evidence=widgets.Accordion(children=[widgets.HTML('<pre style="white-space:pre-wrap">'+html.escape(rec.get('actual_context','(전달 기록 없음)'))+'</pre>')])
        evidence.set_title(0,'실제 전달 컨텍스트')
        value=old.get(rec['key'],{})
        if value.get('answer_sha256')!=rec['answer_sha256'] or value.get('context_sha256')!=rec.get('context_sha256'): value={}
        choice=widgets.Dropdown(description='판정',options=[('선택하세요',''),('충족','pass'),('부분 충족','partial'),('실패','fail'),('실행 오류','error')],value=value.get('verdict',''))
        reason=widgets.Textarea(description='이유',value=value.get('reason',''),layout=widgets.Layout(width='95%',height='90px'))
        forms.append((rec,choice,reason)); panels.append(widgets.VBox([output,evidence,choice,reason])); titles.append(rec['variant'])
    acc=widgets.Accordion(children=panels)
    for i,title in enumerate(titles): acc.set_title(i,title)
    display(acc)
analysis=widgets.Textarea(description='비교 분석',value=previous.get('analysis',''),placeholder='대표 실패·성공 문항 / 원문→검색→전달→생성 / 자동과 사람 차이 / 지연·호출 / 우선 개선',layout=widgets.Layout(width='98%',height='230px'))
decisions={}
for variant in ('rerank','decompose'):
    saved=previous.get('decisions',{}).get(variant,{})
    choice=widgets.Dropdown(description=variant,options=['','유지','수정','제거'],value=saved.get('decision',''))
    reason=widgets.Textarea(description='근거',value=saved.get('reason',''),layout=widgets.Layout(width='95%',height='90px'))
    decisions[variant]=(choice,reason); display(choice,reason)
display(analysis)
button=widgets.Button(description='사람 판정·분석 저장',button_style='success')
message=widgets.Output()
def save_review(_):
    with message:
        clear_output()
        current=PROJECT/json.loads((PROJECT/'results/latest.json').read_text())['relative_path']
        if current!=RUN:
            print('실행 ID가 달라졌습니다. 8번 셀을 다시 실행하세요.'); return
        fresh={r['key']:r for r in read_json(RUN/'records.json',[])}
        reviews=[]
        for rec,choice,reason in forms:
            now=fresh.get(rec['key'])
            if not now or now['answer_sha256']!=rec['answer_sha256'] or now.get('context_sha256')!=rec.get('context_sha256'):
                print('답변/컨텍스트가 바뀌었습니다. 8번 셀을 다시 실행하세요.'); return
            if not choice.value and not reason.value.strip(): continue
            if not choice.value or not reason.value.strip(): print(rec['key'],'판정과 이유를 함께 입력하세요.'); return
            if rec['status']!='completed' and choice.value!='error': print(rec['key'],'는 실행 오류로 판정하세요.'); return
            reviews.append({'key':rec['key'],'answer_sha256':rec['answer_sha256'],'context_sha256':rec.get('context_sha256'),'verdict':choice.value,'reason':reason.value.strip()})
        decision_data={}
        for name,(choice,reason) in decisions.items():
            if bool(choice.value)!=bool(reason.value.strip()): print(name,'판단과 근거를 함께 입력하세요.'); return
            decision_data[name]={'decision':choice.value,'reason':reason.value.strip()}
        data={'run_id':RUN.name,'reviews':reviews,'decisions':decision_data,'analysis':analysis.value.strip()}
        (RUN/'human_review.json').write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
        run_stage('reporting.py','export',needs_env=False)
        print('저장 완료:',len(reviews),'/15. 분석·두 기법 판단도 완료했는지 확인 후 9번을 실행하세요.')
button.on_click(save_review)
display(button,message)


## 9. 최종 README·노트북·결과 ZIP 다운로드

모델을 다시 호출하지 않습니다. 현재 노트북 사본 자동 저장에 실패하면 파일 메뉴에서 `.ipynb`도 별도 다운로드하세요.


In [ ]:
import sys, json, subprocess
from pathlib import Path
from google.colab import files
import json,subprocess,sys
from pathlib import Path
MARKER=Path('/content/week04_gpu_project_path.txt')
if not MARKER.is_file(): raise RuntimeError('먼저 1번 저장 위치 셀을 실행하세요.')
PROJECT=Path(MARKER.read_text().strip())
if not PROJECT.is_dir(): raise RuntimeError('저장 위치에 접근할 수 없습니다. 1번에서 Drive 연결을 확인하세요.')
PY=Path('/content/week04_env/bin/python')
def run_stage(script,*args,needs_env=True):
    if not (PROJECT/'src'/script).is_file(): raise RuntimeError('2번 자료 복원 셀을 실행하세요.')
    if needs_env and not PY.is_file(): raise RuntimeError('3번 환경 설치 셀을 실행하세요.')
    command=[str(PY) if needs_env else sys.executable,'-u',str(PROJECT/'src'/script),*args]
    p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait(): raise RuntimeError('실행 실패. 위의 실제 오류를 확인하세요. 저장 결과는 보존됩니다.')
if not (PROJECT/'results/latest.json').exists():
    raise RuntimeError('현재 경로에 결과가 없습니다. 기존 Drive 연결/다운로드 ZIP을 확인하세요. 1~3번을 무작정 재실행하지 마세요.')
try:
    from google.colab import _message
    notebook=_message.blocking_request('get_ipynb',timeout_sec=30)['ipynb']
    for cell in notebook.get('cells',[]):
        if cell.get('cell_type')=='code': cell['outputs']=[]; cell['execution_count']=None
    (PROJECT/'gemma_rag_week04_gpu_clean.ipynb').write_text(json.dumps(notebook,ensure_ascii=False),encoding='utf-8')
except Exception as exc:
    print('노트북 사본 저장 실패:',type(exc).__name__,'— 파일 메뉴에서 .ipynb도 별도로 내려받으세요.')
subprocess.run([sys.executable,str(PROJECT/'src/reporting.py'),'export'],check=True)
run=PROJECT/json.loads((PROJECT/'results/latest.json').read_text())['relative_path']
summary=json.loads((run/'summary.json').read_text())
print('사람 판정:',summary['human_reviewed_n'],'/15. 미완료라면 백업용 ZIP입니다.')
files.download((PROJECT/'last_export.txt').read_text().strip())
